# LeetCode 2882 — Drop Duplicate Rows bằng Python (pandas)

Link đề: https://leetcode.com/problems/drop-duplicate-rows/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def dropDuplicateEmails(customers: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong bảng `customers`, các hàng có **cùng `email`** là trùng lặp; hãy **chỉ giữ lại hàng xuất hiện đầu tiên** của mỗi email.
2. Hiểu "trùng lặp" theo **một nhóm cột** (`subset`), phân biệt với "trùng cả hàng", và tham số `keep` (`first`, `last`, `False`).
3. Đi từ cách **vòng lặp + `set` những email đã thấy** → **`drop_duplicates(subset="email")`** → **mặt nạ `~duplicated()`** → **`groupby(...).head(1)`** → **chọn nhãn từ `Series.drop_duplicates()`**.
4. Biết **index gốc được giữ** sau khi bỏ hàng (có "lỗ hổng" trong nhãn) và khi nào cần `reset_index(drop=True)`.
5. Tránh các lỗi kinh điển: quên `subset` (không bỏ hàng nào), `inplace=True` rồi `return` kết quả `None`, `keep` sai, sắp xếp trước khi bỏ trùng, dùng `groupby().first()` (trộn dữ liệu và làm mất index).

## 2. Đề bài & ví dụ

Có một bảng `customers` với các cột `customer_id`, `name`, `email`. Vài khách hàng có **cùng email**.

Hãy viết hàm **xóa các hàng trùng lặp dựa trên cột `email`**, chỉ **giữ lại hàng đầu tiên** của mỗi email.

**Ví dụ**

- Input: DataFrame `customers`

| customer_id | name | email |
|---|---|---|
| 1 | Ella | emily@example.com |
| 2 | David | michael@example.com |
| 3 | Zachary | sarah@example.com |
| 4 | Alice | john@example.com |
| 5 | Finn | john@example.com |
| 6 | Violet | alice@example.com |

- Output:

| customer_id | name | email |
|---|---|---|
| 1 | Ella | emily@example.com |
| 2 | David | michael@example.com |
| 3 | Zachary | sarah@example.com |
| 4 | Alice | john@example.com |
| 6 | Violet | alice@example.com |

- Giải thích: Alice (id 4) và Finn (id 5) có cùng email `john@example.com`; ta giữ **Alice** (xuất hiện trước) và bỏ **Finn**.

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc. Việc kết quả có đánh lại nhãn hàng hay không cũng nên đối chiếu với đề gốc; ở notebook này ta giữ nhãn gốc, đúng mặc định của `drop_duplicates`.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6],
    "name": ["Ella", "David", "Zachary", "Alice", "Finn", "Violet"],
    "email": ["emily@example.com", "michael@example.com", "sarah@example.com",
              "john@example.com", "john@example.com", "alice@example.com"],
})
print(customers)
print(customers["email"].value_counts())

   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
4            5     Finn     john@example.com
5            6   Violet    alice@example.com
email
john@example.com       2
emily@example.com      1
michael@example.com    1
sarah@example.com      1
alice@example.com      1
Name: count, dtype: int64


## 3. Kiến thức nền

### 3.1. "Trùng lặp" là gì?

Hai hàng là **trùng lặp** nếu chúng có **cùng giá trị ở những cột đang xét**. Ở đây chỉ xét cột `email`, nên Alice và Finn là trùng (dù `customer_id` và `name` khác nhau). Nếu xét **mọi cột**, sáu hàng này đều khác nhau, không có hàng trùng.

### 3.2. `Series.duplicated()` — đánh dấu hàng trùng

`s.duplicated()` trả về Series `True/False`: `True` ở những phần tử **đã xuất hiện trước đó** (mặc định `keep="first"`: lần xuất hiện đầu tiên là `False`, các lần sau là `True`).

In [2]:
e = customers["email"]
print(e.duplicated().tolist())                 # [False, False, False, False, True, False]
print(e.duplicated(keep="last").tolist())      # lần cuối là False, các lần trước là True
print(e.duplicated(keep=False).tolist())       # mọi hàng thuộc nhóm trùng đều là True

[False, False, False, False, True, False]
[False, False, False, True, False, False]
[False, False, False, True, True, False]


### 3.3. `DataFrame.duplicated(subset=...)`

Với DataFrame, `df.duplicated(subset="email")` xét trùng **chỉ theo cột `email`**. Không đưa `subset` thì xét **mọi cột**.

In [3]:
print(customers.duplicated(subset="email").tolist())
print(customers.duplicated().tolist())          # xét mọi cột -> không hàng nào trùng

[False, False, False, False, True, False]
[False, False, False, False, False, False]


### 3.4. `~` đảo mask và lọc hàng

`~mask` đảo `True ↔ False`. `df[~df.duplicated(subset="email")]` giữ lại các hàng **không** phải "lần xuất hiện sau".

In [4]:
mask = ~customers.duplicated(subset="email")
print(mask.tolist())
print(customers[mask])

[True, True, True, True, False, True]
   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
5            6   Violet    alice@example.com


### 3.5. `DataFrame.drop_duplicates(subset=..., keep=...)`

Gói gọn hai bước trên. Tham số:

- `subset`: cột (hoặc list cột) dùng để xét trùng; bỏ trống = mọi cột.
- `keep`: `"first"` (mặc định, giữ hàng đầu), `"last"` (giữ hàng cuối), `False` (bỏ **tất cả** hàng trùng).
- `inplace`: `True` sửa tại chỗ và **trả `None`**.
- `ignore_index=True`: đánh lại nhãn `0..n-1`.

In [5]:
print(customers.drop_duplicates(subset="email").index.tolist())                 # [0, 1, 2, 3, 5]
print(customers.drop_duplicates(subset="email", keep="last").index.tolist())    # [0, 1, 2, 4, 5]
print(customers.drop_duplicates(subset="email", keep=False).index.tolist())     # [0, 1, 2, 5]
print(customers.drop_duplicates(subset="email", ignore_index=True).index.tolist())   # [0, 1, 2, 3, 4]

[0, 1, 2, 3, 5]
[0, 1, 2, 4, 5]
[0, 1, 2, 5]
[0, 1, 2, 3, 4]


### 3.6. Index gốc sau khi bỏ hàng

Bỏ hàng **không** đánh lại nhãn. Kết quả có nhãn `0, 1, 2, 3, 5` (thiếu `4`). Muốn nhãn liên tục thì `reset_index(drop=True)`. Đề này chỉ yêu cầu bỏ hàng; nhãn gốc giữ nguyên là hợp lý.

In [6]:
res = customers.drop_duplicates(subset="email")
print(res.index.tolist())
print(res.reset_index(drop=True).index.tolist())

[0, 1, 2, 3, 5]
[0, 1, 2, 3, 4]


### 3.7. `set` để nhớ giá trị đã thấy

`set` lưu các phần tử **không trùng** và kiểm tra `x in s` nhanh (`O(1)` trung bình). Dùng nó khi tự duyệt: "email đã gặp rồi thì bỏ hàng; chưa gặp thì giữ và ghi nhớ".

In [7]:
seen = set()
for em in ["a", "b", "a", "c", "b"]:
    if em in seen:
        print(em, "-> đã thấy, bỏ")
    else:
        seen.add(em)
        print(em, "-> mới, giữ")

a -> mới, giữ
b -> mới, giữ
a -> đã thấy, bỏ
c -> mới, giữ
b -> đã thấy, bỏ


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Đi từ trên xuống, giữ một "sổ" các email đã gặp:

| hàng | email | đã có trong sổ? | hành động | sổ sau đó |
|---|---|---|---|---|
| 0 Ella | emily@ | chưa | **giữ** | {emily@} |
| 1 David | michael@ | chưa | **giữ** | {emily@, michael@} |
| 2 Zachary | sarah@ | chưa | **giữ** | {…, sarah@} |
| 3 Alice | john@ | chưa | **giữ** | {…, john@} |
| 4 Finn | john@ | **có rồi** | **bỏ** | không đổi |
| 5 Violet | alice@ | chưa | **giữ** | {…, alice@} |

Kết quả: các hàng `0, 1, 2, 3, 5`. Vì ta đi **từ đầu xuống**, hàng xuất hiện **trước** luôn được giữ (đúng "giữ hàng đầu tiên").

Lưu ý: so sánh bằng **email**, không so cả hàng; và `name` khác nhau không làm hai hàng "khác nhau" theo đề.

## 5. Cách 1 — Brute Force / baseline: vòng lặp với `set` những email đã thấy

Duyệt từng hàng theo thứ tự. Nếu email **chưa** thấy → giữ (ghi lại nhãn và hàng) và thêm vào `seen`. Cuối cùng dựng DataFrame từ các hàng được giữ, **giữ nhãn gốc**.

In [8]:
def drop_dup_loop(customers: pd.DataFrame) -> pd.DataFrame:
    seen = set()
    labels, rows = [], []
    for label, row in zip(customers.index, customers.itertuples(index=False, name=None)):
        email = row[list(customers.columns).index("email")]       # giá trị cột email của hàng này
        if email in seen:
            continue
        seen.add(email)
        labels.append(label)
        rows.append(row)
    return pd.DataFrame(rows, index=labels, columns=customers.columns)

out = drop_dup_loop(customers)
print(out)
print(out.dtypes.tolist())

   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
5            6   Violet    alice@example.com
[dtype('int64'), <StringDtype(storage='python', na_value=nan)>, <StringDtype(storage='python', na_value=nan)>]


**Độ phức tạp:** duyệt `m` hàng, mỗi hàng kiểm tra / thêm `set` `O(1)` trung bình: **Time `O(m)`**, **Space `O(u)`** với `u` là số email khác nhau (cộng bảng kết quả). Nhưng chạy bằng **vòng lặp Python**, chậm với bảng lớn.

**Nút thắt:** ta tự lo nhãn, dtype, thứ tự cột. Việc "bỏ hàng trùng" quá phổ biến nên pandas có sẵn `drop_duplicates` (viết bằng mã C, nhanh).

## 6. Ý tưởng thuật toán / bước đột phá

pandas gói đúng thao tác ta vừa làm bằng tay:

`customers.drop_duplicates(subset="email")`

- `subset="email"`: chỉ xét cột `email`;
- `keep="first"` (mặc định): giữ **hàng đầu tiên** của mỗi nhóm trùng;
- thứ tự hàng và nhãn gốc được giữ.

Các cách khác cho cùng kết quả: dùng `duplicated` làm mask (Cách 3), dùng `groupby(...).head(1)` (Cách 4), hoặc lấy **nhãn** của các email không trùng rồi chọn hàng (Cách 5). Biết nhiều cách giúp bạn linh hoạt khi đề đổi nhẹ (ví dụ giữ hàng **cuối**, hoặc bỏ **tất cả** hàng trùng).

## 7. Các cách tiếp theo

### Cách 2 — `drop_duplicates(subset="email")` (cách chuẩn)

**Giải thích.** Một lời gọi, không sửa bảng đầu vào (mặc định `inplace=False`), trả về bảng mới đã bỏ hàng trùng.

In [9]:
def drop_dup_builtin(customers: pd.DataFrame) -> pd.DataFrame:
    return customers.drop_duplicates(subset="email")

print(drop_dup_builtin(customers))

   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
5            6   Violet    alice@example.com


In [10]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def dropDuplicateEmails(customers: pd.DataFrame) -> pd.DataFrame:
    return customers.drop_duplicates(subset="email", keep="first")

print(dropDuplicateEmails(customers))

   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
5            6   Violet    alice@example.com


**Độ phức tạp Cách 2:** **Time `O(m)`** trung bình (băm các giá trị email), **Space `O(m)`** cho bảng kết quả và cấu trúc băm.

---

### Cách 3 — Mặt nạ `~duplicated(subset="email")`

**Giải thích.** `duplicated(subset="email")` đánh dấu các hàng "lần xuất hiện sau"; `~` đảo lại; lọc bảng bằng mask. Cách này cho bạn **nhìn thấy mask**, tiện kiểm tra và dễ kết hợp thêm điều kiện (`mask & (df["name"] != "")`).

In [11]:
def drop_dup_mask(customers: pd.DataFrame) -> pd.DataFrame:
    is_repeat = customers.duplicated(subset="email")        # True ở lần xuất hiện sau
    return customers[~is_repeat]

print(drop_dup_mask(customers))

   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
5            6   Violet    alice@example.com


In [12]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def dropDuplicateEmails(customers: pd.DataFrame) -> pd.DataFrame:
    return customers.loc[~customers["email"].duplicated()]

print(dropDuplicateEmails(customers))

   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
5            6   Violet    alice@example.com


**Độ phức tạp Cách 3:** **Time `O(m)`**, **Space `O(m)`** (mask `m` phần tử bool + kết quả).

---

### Cách 4 — `groupby("email").head(1)`

**Giải thích.** Chia bảng thành các nhóm theo `email`, lấy **hàng đầu tiên** của mỗi nhóm bằng `head(1)`. `dropna=False` giữ lại cả nhóm có email thiếu (mặc định `groupby` **bỏ** các hàng có khóa `NaN`). Với `sort=False` và `head`, kết quả **giữ thứ tự hàng và nhãn gốc** (khác `first()`, xem Lỗi 6). Dùng được khi cần "n hàng đầu của mỗi nhóm".

In [13]:
def drop_dup_groupby(customers: pd.DataFrame) -> pd.DataFrame:
    return customers.groupby("email", sort=False, dropna=False).head(1)

print(drop_dup_groupby(customers))

   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
5            6   Violet    alice@example.com


In [14]:
# Code nộp LeetCode cho Cách 4
import pandas as pd

def dropDuplicateEmails(customers: pd.DataFrame) -> pd.DataFrame:
    return customers.groupby("email", sort=False, dropna=False).head(1)

print(dropDuplicateEmails(customers))

   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
5            6   Violet    alice@example.com


**Độ phức tạp Cách 4:** **Time `O(m)`** trung bình (băm để gom nhóm), **Space `O(m)`**; hằng số lớn hơn `drop_duplicates`.

---

### Cách 5 — Lấy nhãn từ `Series.drop_duplicates()` rồi chọn bằng `loc`

**Giải thích.** `customers["email"].drop_duplicates()` là Series các email **không trùng** (giữ lần đầu) với **nhãn hàng gốc**. Lấy `.index` rồi `customers.loc[nhãn]`. Cách này chỉ cần "bỏ trùng" trên **một cột** nhỏ, rồi kéo cả hàng về. Cần index **không trùng nhãn** (nếu có nhãn trùng, `loc` sẽ lặp hàng).

In [15]:
def drop_dup_labels(customers: pd.DataFrame) -> pd.DataFrame:
    keep_labels = customers["email"].drop_duplicates().index
    return customers.loc[keep_labels]

print(drop_dup_labels(customers))
print(customers["email"].drop_duplicates())

   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
5            6   Violet    alice@example.com
0      emily@example.com
1    michael@example.com
2      sarah@example.com
3       john@example.com
5      alice@example.com
Name: email, dtype: str


**Độ phức tạp Cách 5:** **Time `O(m)`**, **Space `O(m)`**.

## 8. TRACE từng bước

In DataFrame ở **từng bước**: bảng gốc, mask `duplicated`, bảng sau khi lọc, và trạng thái vòng lặp của baseline.

In [16]:
def trace_builtin(customers):
    print("BƯỚC 0 - bảng gốc", customers.shape)
    print(customers)

    is_repeat = customers.duplicated(subset="email")
    print("\nBƯỚC 1 - duplicated(subset='email'):")
    print(pd.DataFrame({"email": customers["email"], "is_repeat": is_repeat}))

    keep_mask = ~is_repeat
    print("\nBƯỚC 2 - ~is_repeat (True = giữ):", keep_mask.tolist())

    result = customers[keep_mask]
    print("\nBƯỚC 3 - customers[~is_repeat]:", result.shape)
    print(result)
    print("\nnhãn còn lại:", result.index.tolist(), "(thiếu nhãn 4)")

    same = result.equals(customers.drop_duplicates(subset="email"))
    print("giống drop_duplicates(subset='email')?", same)
    return result

_ = trace_builtin(customers)

BƯỚC 0 - bảng gốc (6, 3)
   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
4            5     Finn     john@example.com
5            6   Violet    alice@example.com

BƯỚC 1 - duplicated(subset='email'):
                 email  is_repeat
0    emily@example.com      False
1  michael@example.com      False
2    sarah@example.com      False
3     john@example.com      False
4     john@example.com       True
5    alice@example.com      False

BƯỚC 2 - ~is_repeat (True = giữ): [True, True, True, True, False, True]

BƯỚC 3 - customers[~is_repeat]: (5, 3)
   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
5            6   Violet    alice@example.com



In [17]:
def trace_loop(customers):
    seen = set()
    for label, row in zip(customers.index, customers.itertuples(index=False, name=None)):
        email = row[2]
        action = "bỏ (đã thấy)" if email in seen else "giữ"
        print(f"nhãn {label}: {row[1]:<8} {email:<22} seen có {len(seen)} email -> {action}")
        seen.add(email)

trace_loop(customers)

nhãn 0: Ella     emily@example.com      seen có 0 email -> giữ
nhãn 1: David    michael@example.com    seen có 1 email -> giữ
nhãn 2: Zachary  sarah@example.com      seen có 2 email -> giữ
nhãn 3: Alice    john@example.com       seen có 3 email -> giữ
nhãn 4: Finn     john@example.com       seen có 4 email -> bỏ (đã thấy)
nhãn 5: Violet   alice@example.com      seen có 4 email -> giữ


Bảng tóm tắt hình dạng:

| bước | biểu thức | shape / kết quả |
|---|---|---|
| 0 | `customers` | `(6, 3)` |
| 1 | `duplicated(subset="email")` | mask `[F, F, F, F, T, F]` |
| 2 | `~mask` | `[T, T, T, T, F, T]` |
| 3 | `customers[~mask]` | `(5, 3)`, nhãn `[0, 1, 2, 3, 5]` |

## 9. Code nộp LeetCode

In [18]:
import pandas as pd

def dropDuplicateEmails(customers: pd.DataFrame) -> pd.DataFrame:
    return customers.drop_duplicates(subset="email", keep="first")

result = dropDuplicateEmails(customers)
print(result)
result

   customer_id     name                email
0            1     Ella    emily@example.com
1            2    David  michael@example.com
2            3  Zachary    sarah@example.com
3            4    Alice     john@example.com
5            6   Violet    alice@example.com


,customer_id,name,email
0,1,Ella,emily@example.com
1,2,David,michael@example.com
2,3,Zachary,sarah@example.com
3,4,Alice,john@example.com
5,6,Violet,alice@example.com


## 10. Vì sao thuật toán đúng

- **Xác định trùng đúng:** `drop_duplicates(subset="email")` coi hai hàng là trùng khi **giá trị cột `email` bằng nhau**, bất kể các cột khác.
- **Giữ hàng đầu:** với `keep="first"`, trong mỗi nhóm cùng email, hàng ở **vị trí nhỏ nhất** được giữ, các hàng sau bị bỏ. Đúng yêu cầu "giữ lần xuất hiện đầu tiên".
- **Không bỏ nhầm:** hàng có email chỉ xuất hiện **một lần** thì không bao giờ bị coi là trùng, nên luôn được giữ.
- **Thứ tự và nhãn:** thao tác chỉ **xóa** hàng, không sắp xếp lại; thứ tự tương đối và nhãn gốc của các hàng còn lại không đổi; mọi cột giữ nguyên.
- **Tính chất kết quả:** sau khi bỏ trùng, mọi email trong kết quả là **duy nhất**, và tập email của kết quả **bằng** tập email của bảng gốc (không mất email nào).
- **Giá trị thiếu:** pandas coi các `NaN` / `None` là **bằng nhau** khi xét trùng, nên chỉ giữ lại một hàng có email thiếu đầu tiên.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng, `c` số cột, `u` số email khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp + `set` | `O(m)` (Python thuần, chậm) | `O(u·c)` |
| 2. `drop_duplicates` | `O(m)` trung bình | `O(m)` |
| 3. `~duplicated` | `O(m)` trung bình | `O(m)` |
| 4. `groupby(...).head(1)` | `O(m)` trung bình (hằng số lớn hơn) | `O(m)` |
| 5. `Series.drop_duplicates().index` + `loc` | `O(m)` trung bình | `O(m)` |

"Trung bình" vì việc băm (hash) các chuỗi email. Với `drop_duplicates`, pandas dùng bảng băm trong mã C nên nhanh hơn vòng lặp Python nhiều lần.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [19]:
impls = {
    "loop": drop_dup_loop,
    "builtin": drop_dup_builtin,
    "mask": drop_dup_mask,
    "groupby": drop_dup_groupby,
    "labels": drop_dup_labels,
    "submit": dropDuplicateEmails,
}

expected_example = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 6],
    "name": ["Ella", "David", "Zachary", "Alice", "Violet"],
    "email": ["emily@example.com", "michael@example.com", "sarah@example.com",
              "john@example.com", "alice@example.com"],
}, index=[0, 1, 2, 3, 5])
for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(customers), expected_example, check_index_type=False)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


In [20]:
def make(emails, names=None):
    n = len(emails)
    return pd.DataFrame({"customer_id": list(range(1, n + 1)),
                         "name": names or [f"c{i}" for i in range(n)],
                         "email": emails})

edge = {
    "không trùng": (make(["a", "b", "c"]), [0, 1, 2]),
    "tất cả cùng email": (make(["a", "a", "a", "a"]), [0]),
    "một hàng": (make(["a"]), [0]),
    "trùng xen kẽ": (make(["a", "b", "a", "b", "c", "a"]), [0, 1, 4]),
    "trùng liên tiếp ở cuối": (make(["a", "b", "b", "b"]), [0, 1]),
    "email thiếu (None) coi là trùng nhau, giữ None đầu": (make(["a", None, None, "b"]), [0, 1, 3]),
}
for name, (df, keep_labels) in edge.items():
    for key, fn in impls.items():
        got = fn(df)
        assert got.index.tolist() == keep_labels, (name, key, got.index.tolist())
        pd.testing.assert_frame_equal(got, df.loc[keep_labels], check_index_type=False,
                                      check_dtype=("None" not in name))
    print("PASS", name, "-> giữ nhãn", keep_labels)

PASS không trùng -> giữ nhãn [0, 1, 2]
PASS tất cả cùng email -> giữ nhãn [0]
PASS một hàng -> giữ nhãn [0]
PASS trùng xen kẽ -> giữ nhãn [0, 1, 4]
PASS trùng liên tiếp ở cuối -> giữ nhãn [0, 1]
PASS email thiếu (None) coi là trùng nhau, giữ None đầu -> giữ nhãn [0, 1, 3]


### 12.2. Tính chất của kết quả, đầu vào không bị sửa, và phản ví dụ

In [21]:
res = dropDuplicateEmails(customers)
assert res["email"].is_unique                                   # email duy nhất
assert set(res["email"]) == set(customers["email"])             # không mất email nào
assert res["customer_id"].tolist() == sorted(res["customer_id"].tolist())   # giữ thứ tự gốc
before = customers.copy()
for key, fn in impls.items():
    fn(customers)
pd.testing.assert_frame_equal(customers, before)                # đầu vào không đổi
print("PASS tính chất kết quả và không sửa đầu vào")

# Phản ví dụ 1: không có subset -> xét mọi cột -> không bỏ hàng nào
assert len(customers.drop_duplicates()) == 6
# Phản ví dụ 2: keep='last' giữ Finn (id 5) thay vì Alice (id 4)
last = customers.drop_duplicates(subset="email", keep="last")
assert 5 in last["customer_id"].tolist() and 4 not in last["customer_id"].tolist()
# Phản ví dụ 3: keep=False bỏ CẢ HAI hàng john@example.com
none_left = customers.drop_duplicates(subset="email", keep=False)
assert 4 not in none_left["customer_id"].tolist() and 5 not in none_left["customer_id"].tolist()
print("PASS các phản ví dụ subset / keep")

PASS tính chất kết quả và không sửa đầu vào
PASS các phản ví dụ subset / keep


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh bảng ngẫu nhiên `m ∈ [1, 80]` với email chọn từ một nhóm nhỏ (để nhiều email trùng), index có thể lộn xộn. Đáp án chuẩn dựng bằng từ điển "email → nhãn đầu tiên".

In [22]:
import random
random.seed(2882)
np.random.seed(2882)
pool = ["a@x.com", "b@x.com", "c@x.com", "d@x.com", "e@x.com", "f@x.com"]

for t in range(300):
    m = random.randint(1, 80)
    k = random.randint(1, len(pool))
    df = pd.DataFrame({
        "customer_id": np.arange(1, m + 1),
        "name": [random.choice(["An", "Binh", "Chau"]) for _ in range(m)],
        "email": [random.choice(pool[:k]) for _ in range(m)],
    })
    if random.random() < 0.5:
        df.index = random.sample(range(1000), m)
    first_label = {}
    for label, em in zip(df.index, df["email"]):
        first_label.setdefault(em, label)               # nhãn đầu tiên của mỗi email
    exp_labels = [lab for lab in df.index if lab in set(first_label.values())]
    for key, fn in impls.items():
        got = fn(df)
        assert got.index.tolist() == exp_labels, (t, key)
        assert got["email"].is_unique
        assert got["customer_id"].tolist() == df.loc[exp_labels, "customer_id"].tolist()
print("300 bảng ngẫu nhiên: mọi cách giữ đúng hàng đầu tiên của mỗi email")

300 bảng ngẫu nhiên: mọi cách giữ đúng hàng đầu tiên của mỗi email


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `subset` — không bỏ hàng nào

In [23]:
sai = customers.drop_duplicates()                         # xét mọi cột
print("sai  :", len(sai), "hàng (không bỏ hàng nào vì customer_id khác nhau)")
print("đúng :", len(dropDuplicateEmails(customers)), "hàng")

sai  : 6 hàng (không bỏ hàng nào vì customer_id khác nhau)
đúng : 5 hàng


**Vì sao sai:** không có `subset`, pandas chỉ bỏ hàng **giống hệt ở mọi cột**. Alice và Finn khác `customer_id` và `name` nên không bị coi là trùng. Đề yêu cầu trùng theo **email**.

### Lỗi 2: `inplace=True` rồi `return` kết quả của nó

In [24]:
def sai_inplace(customers):
    return customers.drop_duplicates(subset="email", inplace=True)    # sai: trả về None

print("sai  :", sai_inplace(customers.copy()))
print("đúng :", type(dropDuplicateEmails(customers)).__name__)

sai  : None
đúng : DataFrame


**Vì sao sai:** với `inplace=True`, hàm **sửa tại chỗ** và **trả `None`**. Nếu muốn dùng `inplace=True` thì phải `return customers` ở dòng sau; còn tốt nhất là không dùng `inplace`.

### Lỗi 3: Nhầm `keep`

In [25]:
sai = customers.drop_duplicates(subset="email", keep="last")
print("keep='last' giữ id:", sai["customer_id"].tolist())            # giữ Finn(5) thay vì Alice(4)
print("đề cần (first) id: ", dropDuplicateEmails(customers)["customer_id"].tolist())

keep='last' giữ id: [1, 2, 3, 5, 6]
đề cần (first) id:  [1, 2, 3, 4, 6]


**Vì sao sai:** đề cần giữ hàng **đầu tiên**, tức `keep="first"` (mặc định). `keep="last"` giữ hàng cuối; `keep=False` bỏ **tất cả** hàng thuộc nhóm trùng.

### Lỗi 4: Sắp xếp trước khi bỏ trùng

In [26]:
sai = customers.sort_values("email").drop_duplicates(subset="email")
print("sai  nhãn:", sai.index.tolist(), "(thứ tự hàng bị đổi theo email)")
print("đúng nhãn:", dropDuplicateEmails(customers).index.tolist())
assert sai.index.tolist() != dropDuplicateEmails(customers).index.tolist()

sai  nhãn: [5, 0, 3, 1, 2] (thứ tự hàng bị đổi theo email)
đúng nhãn: [0, 1, 2, 3, 5]


**Vì sao sai:** `sort_values` **đổi thứ tự hàng** nên "đầu tiên" không còn là đầu tiên theo bảng gốc, và kết quả bị đổi thứ tự. Không sắp xếp nếu đề không yêu cầu.

### Lỗi 5: Quên `.reset_index` khi đề cần nhãn liên tục (và ngược lại)

In [27]:
res = dropDuplicateEmails(customers)
print("nhãn mặc định     :", res.index.tolist())                          # [0, 1, 2, 3, 5]
print("sau reset_index   :", res.reset_index(drop=True).index.tolist())   # [0, 1, 2, 3, 4]
print("equals có xét nhãn:", res.equals(res.reset_index(drop=True)))      # False

nhãn mặc định     : [0, 1, 2, 3, 5]
sau reset_index   : [0, 1, 2, 3, 4]
equals có xét nhãn: False


**Cẩn thận:** hai bảng cùng dữ liệu nhưng **khác nhãn** thì `equals` trả `False`. Khi so với đáp án, hãy xem đề muốn nhãn gốc hay nhãn đánh lại (đối chiếu lại với đề gốc). Đừng gọi `reset_index` nếu không cần.

### Lỗi 6: Dùng `groupby("email").first()`

In [28]:
sai = customers.groupby("email", sort=False, as_index=False).first()
print(sai.columns.tolist())                 # cột email đứng đầu, thứ tự cột đổi
print(sai.index.tolist())                   # nhãn bị đánh lại 0..4
m = pd.DataFrame({"email": ["x", "x"], "name": [None, "B"], "k": [1, 2]})
print(m.groupby("email").first())           # first() lấy giá trị KHÔNG thiếu đầu tiên của TỪNG cột

['email', 'customer_id', 'name']
[0, 1, 2, 3, 4]
      name  k
email        
x        B  1


**Vì sao sai:** `GroupBy.first()` trả về giá trị **không thiếu đầu tiên của từng cột**, nên có thể **trộn dữ liệu từ nhiều hàng** (ở trên `name = B` lấy từ hàng thứ hai nhưng `k = 1` từ hàng đầu). Nó còn đổi thứ tự cột và đánh lại nhãn. Dùng `head(1)` hoặc `drop_duplicates` để giữ **nguyên hàng**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xóa hàng trùng / giữ bản ghi đầu tiên" → `df.drop_duplicates(subset=[cột], keep="first")`.
- Trùng theo **vài cột** → `subset=["a", "b"]`. Trùng theo **mọi cột** → bỏ `subset`.
- Muốn **xem** hàng nào trùng → `df[df.duplicated(subset=[cột], keep=False)]`.
- Muốn **giữ bản mới nhất** → sắp xếp theo thời gian rồi `keep="last"` (hoặc `sort_values(...).drop_duplicates(..., keep="last")`).
- Muốn nhãn liên tục → `ignore_index=True` hoặc `reset_index(drop=True)`; nếu đề không nói, giữ nhãn gốc.
- Đừng `inplace=True` rồi `return` kết quả; cứ `return df.drop_duplicates(...)`.

## 15. Biến thể của bài

1. **Giữ hàng cuối cùng:** `keep="last"`.
2. **Bỏ tất cả hàng trùng (chỉ giữ email duy nhất):** `keep=False`.
3. **Trùng theo nhiều cột:** `subset=["name", "email"]`.
4. **Đếm số hàng trùng:** `df.duplicated(subset="email").sum()`.
5. **Giữ bản có `customer_id` nhỏ nhất / lớn nhất:** `sort_values("customer_id").drop_duplicates(subset="email")`.
6. **Chuẩn hóa trước khi so:** `df["email"].str.lower().str.strip()` rồi mới bỏ trùng (email viết hoa / thường khác nhau).
7. Liên quan LeetCode 2880 *Select Data*, 182 *Duplicate Emails* (SQL), 196 *Delete Duplicate Emails* (SQL).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp + `set` | nhớ email đã thấy | `O(m)` (Python, chậm) | `O(u·c)` | hiểu bản chất; không dùng trên dữ liệu lớn |
| 2. `drop_duplicates(subset=)` | hàm có sẵn | `O(m)` | `O(m)` | **cách nộp bài** |
| 3. `~duplicated(subset=)` | mask rồi lọc | `O(m)` | `O(m)` | muốn thấy mask, kết hợp thêm điều kiện |
| 4. `groupby(...).head(1)` | hàng đầu của từng nhóm | `O(m)` (hằng số lớn) | `O(m)` | khi cần "`n` hàng đầu mỗi nhóm" |
| 5. `Series.drop_duplicates().index` + `loc` | bỏ trùng trên một cột rồi lấy nhãn | `O(m)` | `O(m)` | khi index duy nhất, muốn xử lý cột nhỏ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `pd.Series(["a", "b", "a"]).duplicated().tolist()`, `pd.Series(["a", "b", "a"]).duplicated(keep="last").tolist()`, `len(customers.drop_duplicates())`, `len(customers.drop_duplicates(subset="email"))`.

**Bài 2 (tính tay).** Với `email = ["x", "y", "x", "z", "y"]` (nhãn `0..4`): viết mask `duplicated()` (mặc định) và nhãn các hàng còn lại sau `drop_duplicates()`; rồi làm lại với `keep="last"`.

**Bài 3 (điền TODO).** Hoàn thành `drop_dup_todo(df)` bằng mask.

In [29]:
def drop_dup_todo(df):
    # TODO 1: is_repeat = df.duplicated(subset="email")
    # TODO 2: return df[~is_repeat]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `drop_dup_loop2(customers)` bằng vòng lặp + `set` (Cách 1) mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `keep_latest(customers)`: với mỗi email giữ hàng có `customer_id` **lớn nhất** (nhãn gốc, thứ tự gốc). (b) Viết `only_unique_emails(customers)`: chỉ giữ các hàng có email **xuất hiện đúng một lần** trong bảng.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[False, False, True]`; `[True, False, False]`; `6`; `5`. Bài 2: mask mặc định đánh dấu lần xuất hiện **sau**.
- **Mức 2 (vừa):** Bài 2: mặc định mask = `[F, F, T, F, T]`, nhãn còn lại `[0, 1, 3]`; với `keep="last"` mask = `[T, T, F, F, F]`, nhãn còn lại `[2, 3, 4]`. Bài 3: hai dòng đã ghi trong comment. Bài 4: `seen = set()`, duyệt từng hàng, bỏ qua nếu email đã trong `seen`.
- **Mức 3 (sâu):** Bài 5(a): `customers.sort_values("customer_id").drop_duplicates(subset="email", keep="last")` rồi `sort_index()` để trả lại thứ tự gốc. Bài 5(b): `customers[~customers.duplicated(subset="email", keep=False)]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert pd.Series(["a", "b", "a"]).duplicated().tolist() == [False, False, True]
assert pd.Series(["a", "b", "a"]).duplicated(keep="last").tolist() == [True, False, False]
assert len(customers.drop_duplicates()) == 6
assert len(customers.drop_duplicates(subset="email")) == 5

# Bài 2
s = pd.Series(["x", "y", "x", "z", "y"])
assert s.duplicated().tolist() == [False, False, True, False, True]
assert s.drop_duplicates().index.tolist() == [0, 1, 3]
assert s.duplicated(keep="last").tolist() == [True, True, False, False, False]
assert s.drop_duplicates(keep="last").index.tolist() == [2, 3, 4]

# Bài 3
def drop_dup_todo_ans(df):
    is_repeat = df.duplicated(subset="email")
    return df[~is_repeat]

pd.testing.assert_frame_equal(drop_dup_todo_ans(customers), expected_example, check_index_type=False)

# Bài 4
def drop_dup_loop2(customers):
    seen = set()
    keep = []
    for label, email in zip(customers.index, customers["email"]):
        if email not in seen:
            seen.add(email)
            keep.append(label)
    return customers.loc[keep]

pd.testing.assert_frame_equal(drop_dup_loop2(customers), expected_example, check_index_type=False)

# Bài 5(a)
def keep_latest(customers):
    return customers.sort_values("customer_id").drop_duplicates(subset="email", keep="last").sort_index()

r5a = keep_latest(customers)
assert r5a["customer_id"].tolist() == [1, 2, 3, 5, 6]          # Finn (5) thay cho Alice (4)

# Bài 5(b)
def only_unique_emails(customers):
    return customers[~customers.duplicated(subset="email", keep=False)]

r5b = only_unique_emails(customers)
assert r5b["customer_id"].tolist() == [1, 2, 3, 6]
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (bỏ hàng trùng theo một cột, giữ hàng đầu)

```python
import pandas as pd

def dropDuplicateEmails(customers: pd.DataFrame) -> pd.DataFrame:
    return customers.drop_duplicates(subset="email", keep="first")
```

**Checklist**

- [ ] Có `subset="email"` (trùng theo email, không phải theo cả hàng)?
- [ ] `keep="first"` (mặc định) đúng yêu cầu "giữ hàng đầu tiên"?
- [ ] **Trả về** DataFrame (không `inplace=True` rồi trả `None`)?
- [ ] Không `sort_values` trước khi bỏ trùng (sẽ đổi hàng được giữ và thứ tự)?
- [ ] Biết nhãn gốc được giữ; chỉ `reset_index(drop=True)` nếu đề yêu cầu?
- [ ] Đã thử bảng không trùng, toàn trùng, email thiếu (`None`)?
- [ ] Skeleton đúng `def dropDuplicateEmails(customers: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?